# Industrial Machine Condition Classification
Models: LogReg, KNN, Decision Tree, Random Forest, XGBoost, CatBoost (+ blend). Metric: Macro-F1 & Accuracy (5-fold Stratified CV)

In [ ]:
# !pip install xgboost catboost scikit-learn pandas numpy -q
import os, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

SEED, N_FOLDS = 42, 5
SCORING = "f1_macro"          # change to "accuracy" if Kaggle metric is accuracy
# Kaggle metric ke hisaab se: "f1_macro" / "accuracy"

In [ ]:
# ---- Load data (folder auto-detect) ----
for d in [".", "industrial-machine-condition-classification-mtech", "data", "/kaggle/input/industrial-machine-condition-classification-mtech"]:
    if os.path.exists(os.path.join(d, "train.csv")):
        DATA_DIR = d; break
else:
    raise FileNotFoundError("train.csv nahi mili - DATA_DIR manually set karo")
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
sub_path = f"{DATA_DIR}/sample_submission.csv"
sample = pd.read_csv(sub_path) if os.path.exists(sub_path) else None

# ID & target auto-detect (sample_submission: col0 = id, col1 = target)
if sample is not None:
    ID_COL, TARGET = sample.columns[0], sample.columns[1]
else:
    TARGET = [c for c in train.columns if c not in test.columns][0]
    ID_COL = test.columns[0]
print("ID:", ID_COL, "| TARGET:", TARGET)
print(train.shape, test.shape)
train.head()

## EDA

In [ ]:
print(train.dtypes.value_counts())
print("\nMissing (top):\n", train.isna().sum().sort_values(ascending=False).head(10))
print("\nDuplicates:", train.duplicated().sum())
print("\nClass distribution:\n", train[TARGET].value_counts(normalize=True))
train.describe().T.head(20)

In [ ]:
# Correlation of numeric features with target (encoded) - quick insight
tmp = train.copy(); tmp[TARGET] = LabelEncoder().fit_transform(tmp[TARGET])
num_tmp = tmp.select_dtypes("number")
print(num_tmp.corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(15))

## Preprocessing

In [ ]:
train = train.drop_duplicates().reset_index(drop=True)
le = LabelEncoder()
y = le.fit_transform(train[TARGET]); n_classes = len(le.classes_)
test_ids = test[ID_COL]

drop_cols = [ID_COL] if ID_COL in train.columns else []
X = train.drop(columns=[TARGET] + drop_cols)
X_test = test.drop(columns=[c for c in [ID_COL] if c in test.columns])[X.columns]

cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print("Numeric:", len(num_cols), "| Categorical:", len(cat_cols), "| Classes:", n_classes)

skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)

# sklearn preprocessing (impute + scale + one-hot)
def make_pre():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ])

## Baseline models (for report table)

In [ ]:
results = {}
sk_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "KNN":                 KNeighborsClassifier(n_neighbors=7, weights="distance"),
    "Decision Tree":       DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=SEED),
    "Random Forest":       RandomForestClassifier(n_estimators=400, n_jobs=-1, class_weight="balanced", random_state=SEED),
}
for name, m in sk_models.items():
    pipe = Pipeline([("pre", make_pre()), ("m", m)])
    cv = cross_validate(pipe, X, y, cv=skf, scoring=["accuracy", "f1_macro"], n_jobs=-1)
    results[name] = {"Accuracy": cv["test_accuracy"].mean(), "Macro-F1": cv["test_f1_macro"].mean()}
    print(f"{name:22s} acc={results[name]['Accuracy']:.4f}  f1={results[name]['Macro-F1']:.4f}")

## XGBoost & CatBoost (5-fold, early stopping, fold-averaged test predictions)

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

# --- XGBoost data (native categorical) ---
Xx, Xx_test = X.copy(), X_test.copy()
for c in cat_cols:
    cats = pd.Categorical(pd.concat([Xx[c], Xx_test[c]]).astype(str)).categories
    Xx[c] = pd.Categorical(Xx[c].astype(str), categories=cats)
    Xx_test[c] = pd.Categorical(Xx_test[c].astype(str), categories=cats)

xgb_params = dict(n_estimators=3000, learning_rate=0.03, max_depth=6, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=2, reg_lambda=1.5,
                  tree_method="hist", enable_categorical=True,
                  early_stopping_rounds=150, random_state=SEED, n_jobs=-1,
                  objective="multi:softprob" if n_classes > 2 else "binary:logistic")

oof_xgb = np.zeros((len(X), n_classes)); test_xgb = np.zeros((len(X_test), n_classes))
for f, (tr, va) in enumerate(skf.split(Xx, y)):
    m = XGBClassifier(**xgb_params)
    m.fit(Xx.iloc[tr], y[tr], sample_weight=compute_sample_weight("balanced", y[tr]),
          eval_set=[(Xx.iloc[va], y[va])], verbose=False)
    p = m.predict_proba(Xx.iloc[va]); p = p if n_classes > 2 else np.c_[1-p[:,1], p[:,1]] if p.ndim==1 else p
    oof_xgb[va] = p
    pt = m.predict_proba(Xx_test); test_xgb += (pt if pt.ndim==2 else np.c_[1-pt, pt]) / N_FOLDS
    print(f"XGB fold {f+1}: f1={f1_score(y[va], p.argmax(1), average='macro'):.4f} | iters={m.best_iteration}")
results["XGBoost"] = {"Accuracy": accuracy_score(y, oof_xgb.argmax(1)), "Macro-F1": f1_score(y, oof_xgb.argmax(1), average="macro")}
print(results["XGBoost"])

In [ ]:
# --- CatBoost (native categorical) ---
Xc, Xc_test = X.copy(), X_test.copy()
for c in cat_cols:
    Xc[c] = Xc[c].astype(str).fillna("missing"); Xc_test[c] = Xc_test[c].astype(str).fillna("missing")

oof_cat = np.zeros((len(X), n_classes)); test_cat = np.zeros((len(X_test), n_classes))
for f, (tr, va) in enumerate(skf.split(Xc, y)):
    m = CatBoostClassifier(iterations=3000, learning_rate=0.05, depth=6, l2_leaf_reg=3,
                           loss_function="MultiClass" if n_classes > 2 else "Logloss",
                           auto_class_weights="Balanced", random_seed=SEED,
                           early_stopping_rounds=150, verbose=0, cat_features=cat_cols)
    m.fit(Xc.iloc[tr], y[tr], eval_set=(Xc.iloc[va], y[va]))
    p = m.predict_proba(Xc.iloc[va]); oof_cat[va] = p
    test_cat += m.predict_proba(Xc_test) / N_FOLDS
    print(f"Cat fold {f+1}: f1={f1_score(y[va], p.argmax(1), average='macro'):.4f} | iters={m.get_best_iteration()}")
results["CatBoost"] = {"Accuracy": accuracy_score(y, oof_cat.argmax(1)), "Macro-F1": f1_score(y, oof_cat.argmax(1), average="macro")}
print(results["CatBoost"])

## Blend + results table

In [ ]:
best_w, best_s = 0.5, -1
for w in np.linspace(0, 1, 21):
    s = f1_score(y, (w*oof_xgb + (1-w)*oof_cat).argmax(1), average="macro")
    if s > best_s: best_w, best_s = w, s
oof_blend = best_w*oof_xgb + (1-best_w)*oof_cat
test_blend = best_w*test_xgb + (1-best_w)*test_cat
results[f"Blend XGB({best_w:.2f})+Cat"] = {"Accuracy": accuracy_score(y, oof_blend.argmax(1)), "Macro-F1": best_s}

res_df = pd.DataFrame(results).T.sort_values("Macro-F1", ascending=False).round(4)
res_df.index.name = "Model"; res_df   # <- yahi table report me daalo

## Submission

In [ ]:
def save_sub(proba, fname):
    out = pd.DataFrame({ID_COL: test_ids, TARGET: le.inverse_transform(proba.argmax(1))})
    out.to_csv(fname, index=False); return out

save_sub(test_xgb,   "submission_xgb.csv")
save_sub(test_cat,   "submission_cat.csv")
final = save_sub(test_blend, "submission_blend.csv")   # usually best -> isko Kaggle pe submit karo
if sample is not None:
    assert list(final.columns) == list(sample.columns) and len(final) == len(sample)
print(final.shape, "| NaN:", final[TARGET].isna().sum())
print(final[TARGET].value_counts(normalize=True))
final.head()